### 📦 CELDA 1 —  Importaciones

In [6]:
# ── CELDA 1: Importaciones ────────────────────────────────────
import sys
import os

# Agrega la raíz del proyecto al path para importar scripts/
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Motor de base de datos reutilizando database.py del proyecto
from scripts.database import engine

# Estilo coherente
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print("✅ Librerías importadas correctamente")
print(f"   pandas  {pd.__version__}")
print(f"   numpy   {np.__version__}")
print(f"   seaborn {sns.__version__}")

✅ Librerías importadas correctamente
   pandas  2.2.0
   numpy   1.26.4
   seaborn 0.13.2


### 📊 CELDA 2 — Carga de datos desde PostgreSQL

In [7]:
# ── CELDA 2: Carga de datos desde PostgreSQL ─────────────────

QUERY = """
SELECT
    s.nombre        AS superheroe,
    s.editorial,
    p.inteligencia,
    p.fuerza,
    p.velocidad,
    p.durabilidad,
    p.combate,
    p.poder,
    p.fecha_extraccion
FROM powerstats p
JOIN superheroes s ON p.superhero_id = s.id
ORDER BY p.fecha_extraccion
"""

df = pd.read_sql(QUERY, engine)

print("✅ Datos cargados exitosamente desde PostgreSQL")
print(f"   Filas    : {df.shape[0]:,}")
print(f"   Columnas : {df.shape[1]}")

print("\n📋 Primeras filas:")
df.head()

✅ Datos cargados exitosamente desde PostgreSQL
   Filas    : 3,535
   Columnas : 9

📋 Primeras filas:


,superheroe,editorial,inteligencia,fuerza,velocidad,durabilidad,combate,poder,fecha_extraccion
0,Iron Man,None,64,80,48,56,69,60,2026-03-27 20:26:54.563881
1,Captain Hindsight,None,66,83,90,68,60,74,2026-03-27 20:36:54.563881
2,Captain Midnight,None,59,72,71,100,63,75,2026-03-27 20:46:54.563881
3,Iron Man,None,63,79,60,63,53,63,2026-03-27 20:56:54.563881
4,Captain Hindsight,None,57,100,95,69,65,86,2026-03-27 21:06:54.563881


### 🔢 CELDA 3 — Variables

In [8]:
# ── CELDA 3: Preparación de variables ───────────────────────

features = ["inteligencia", "fuerza", "velocidad", "durabilidad", "combate"]
target = "poder"

X = df[features].values
y = df[target].values

print("✅ Variables listas")
print(f"   Features usadas: {features}")
print(f"   Variable objetivo: {target}")
print(f"   Shape X: {X.shape}")
print(f"   Shape y: {y.shape}")

✅ Variables listas
   Features usadas: ['inteligencia', 'fuerza', 'velocidad', 'durabilidad', 'combate']
   Variable objetivo: poder
   Shape X: (3535, 5)
   Shape y: (3535,)


### 🧠 🔁 FUNCIÓN PRINCIPAL

### ⚙️ CELDA 4 — Función evaluar_split()

In [10]:
# ── CELDA 4: Función evaluación del modelo ───────────────────

def evaluar_split(X, y, test_size, split_label, resultados_globales):

    # Crear carpeta de salida
    save_dir = "data/graficas/arbol_decision"
    os.makedirs(save_dir, exist_ok=True)

    print(f"\n🚀 Ejecutando split {split_label}")

    # 1. Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42
    )

    print(f"   Train: {len(X_train):,} | Test: {len(X_test):,}")

    # 2. Modelo
    modelo = DecisionTreeRegressor(
        max_depth=4,
        min_samples_leaf=10,
        random_state=42
    )

    modelo.fit(X_train, y_train)

    # 3. Predicciones
    y_pred = modelo.predict(X_test)

    # 4. Métricas
    r2   = r2_score(y_test, y_pred)
    mse  = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_test, y_pred)

    print("📊 Métricas:")
    print(f"   R²   : {r2:.4f}")
    print(f"   RMSE : {rmse:.4f}")
    print(f"   MAE  : {mae:.4f}")

    resultados_globales.append({
        "Split": split_label,
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae
    })

    # ── Gráfico 1: Residuos ──
    residuos = y_test - y_pred

    plt.figure()
    plt.scatter(y_pred, residuos, alpha=0.5)
    plt.axhline(0)
    plt.title(f"Residuos vs Predicción ({split_label})")
    plt.xlabel("Predicción")
    plt.ylabel("Residuo")

    ruta1 = f"{save_dir}/residuos_{split_label}.png"
    plt.savefig(ruta1)
    plt.close()

    # ── Gráfico 2: Real vs Predicho ──
    plt.figure()
    plt.scatter(y_test, y_pred, alpha=0.5)

    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], '--')

    plt.xlabel("Valor real")
    plt.ylabel("Predicción")
    plt.title(f"Real vs Predicho ({split_label})")

    ruta2 = f"{save_dir}/real_vs_pred_{split_label}.png"
    plt.savefig(ruta2)
    plt.close()

    print("🖼️ Gráficas guardadas en:")
    print(f"   {ruta1}")
    print(f"   {ruta2}")

    return modelo, X_train, y_train

### ✂️ SPLITS

### CELDA 5 — Ejecutar 80/20

In [11]:
# ── CELDA 5: Split 80/20 ──────────────────────────────────

resultados = []

modelo_s1, X_train_s1, y_train_s1 = evaluar_split(
    X, y, test_size=0.2, split_label="80_20", resultados_globales=resultados
)


🚀 Ejecutando split 80_20
   Train: 2,828 | Test: 707
📊 Métricas:
   R²   : 0.4260
   RMSE : 5.6419
   MAE  : 4.4672
🖼️ Gráficas guardadas en:
   data/graficas/arbol_decision/residuos_80_20.png
   data/graficas/arbol_decision/real_vs_pred_80_20.png


### CELDA 6 — Ejecutar 70/30

In [12]:
# ── CELDA 6: Split 70/30 ──────────────────────────────────

modelo_s2, _, _ = evaluar_split(
    X, y, test_size=0.3, split_label="70_30", resultados_globales=resultados
)


🚀 Ejecutando split 70_30
   Train: 2,474 | Test: 1,061
📊 Métricas:
   R²   : 0.4031
   RMSE : 5.7517
   MAE  : 4.5746
🖼️ Gráficas guardadas en:
   data/graficas/arbol_decision/residuos_70_30.png
   data/graficas/arbol_decision/real_vs_pred_70_30.png


### CELDA 7 — Ejecutar 60/40

In [14]:
# ── CELDA 7: Split 60/40 ──────────────────────────────────

modelo_s3, _, _ = evaluar_split(
    X, y, test_size=0.4, split_label="60_40", resultados_globales=resultados
)


🚀 Ejecutando split 60_40
   Train: 2,121 | Test: 1,414
📊 Métricas:
   R²   : 0.4001
   RMSE : 5.8497
   MAE  : 4.6435
🖼️ Gráficas guardadas en:
   data/graficas/arbol_decision/residuos_60_40.png
   data/graficas/arbol_decision/real_vs_pred_60_40.png


### 📊 CELDA 8 — Comparación de métricas

In [15]:
# ── CELDA 8: Comparación de resultados ───────────────────

df_resultados = pd.DataFrame(resultados)

print("\n📊 Comparación de splits:")
df_resultados


📊 Comparación de splits:


,Split,R2,RMSE,MAE
0,80_20,0.425962,5.641887,4.467229
1,70_30,0.403081,5.751740,4.574562
2,60_40,0.400084,5.849650,4.643526
3,60_40,0.400084,5.849650,4.643526


### 🌳 VISUALIZACIÓN DEL ÁRBOL

### CELDA 9 — Árbol real (split 80/20)

In [16]:
# ── CELDA 9: Árbol del modelo ─────────────────────

plt.figure(figsize=(22, 10))

plot_tree(
    modelo_s1,
    max_depth=4,
    feature_names=["Inteligencia", "Fuerza", "Velocidad", "Durabilidad", "Combate"],
    filled=True,
    rounded=True,
    impurity=False
)

plt.title("Árbol de Decisión (niveles iniciales)")

ruta = "data/graficas/arbol_decision/arbol_real.png"
plt.savefig(ruta)
plt.close()

print(f"🖼️ Árbol guardado en: {ruta}")

🖼️ Árbol guardado en: data/graficas/arbol_decision/arbol_real.png


### CELDA 10 — Árbol pedagógico

In [17]:
# ── CELDA 10: Árbol pedagógico ───────────────────────────

modelo_viz = DecisionTreeRegressor(max_depth=3, random_state=42)
modelo_viz.fit(X_train_s1, y_train_s1)

plt.figure(figsize=(24, 12))

plot_tree(
    modelo_viz,
    feature_names=["Inteligencia", "Fuerza", "Velocidad", "Durabilidad", "Combate"],
    filled=True,
    rounded=True,
    impurity=False
)

ruta = "data/graficas/arbol_decision/arbol_pedagogico.png"
plt.savefig(ruta)
plt.close()

print(f"🖼️ Árbol pedagógico guardado en: {ruta}")

🖼️ Árbol pedagógico guardado en: data/graficas/arbol_decision/arbol_pedagogico.png


### CELDA 11 — Reglas del árbol

In [18]:
# ── CELDA 11: Reglas del árbol ───────────────────────────

reglas = export_text(
    modelo_viz,
    feature_names=["Inteligencia", "Fuerza", "Velocidad", "Durabilidad", "Combate"]
)

print("📜 Reglas del modelo:\n")
print(reglas)

📜 Reglas del modelo:

|--- Fuerza <= 84.50
|   |--- Velocidad <= 74.50
|   |   |--- Inteligencia <= 88.50
|   |   |   |--- value: [67.41]
|   |   |--- Inteligencia >  88.50
|   |   |   |--- value: [73.52]
|   |--- Velocidad >  74.50
|   |   |--- Inteligencia <= 82.50
|   |   |   |--- value: [72.07]
|   |   |--- Inteligencia >  82.50
|   |   |   |--- value: [78.32]
|--- Fuerza >  84.50
|   |--- Inteligencia <= 82.50
|   |   |--- Combate <= 70.50
|   |   |   |--- value: [73.51]
|   |   |--- Combate >  70.50
|   |   |   |--- value: [77.10]
|   |--- Inteligencia >  82.50
|   |   |--- Inteligencia <= 85.50
|   |   |   |--- value: [82.17]
|   |   |--- Inteligencia >  85.50
|   |   |   |--- value: [87.83]

